# Pharma Document Q&A — Open-Source RAG Pipeline

A complete Retrieval-Augmented Generation system: upload PDFs (digital or scanned),
ask questions, and get grounded answers with cited sources and confidence scores.

**Pipeline:** OCR-aware PDF extraction -> LLM-based document classification & chunking
-> HuggingFace embeddings + LlamaIndex vector index -> query routing -> metadata-filtered
retrieval -> context-grounded prompting -> **Phi-3-mini-4k-instruct** (open-source, not Gemini)
-> answer with sources, confidence, and chunk count -> themed Gradio UI.


## Setup and Installation

In [1]:
# Install Required Packages

# llama-index: RAG framework (vector index, retrieval, metadata filters)
# transformers/accelerate: run Phi-3-mini-4k-instruct locally
# sentence-transformers / llama-index-embeddings-huggingface: open-source embeddings
# PyPDF2: digital PDF text extraction
# pdf2image + pytesseract: OCR for scanned PDFs
# langchain-text-splitters: RecursiveCharacterTextSplitter for chunking
# gradio: the UI
# poppler-utils / tesseract-ocr: system dependencies pdf2image and pytesseract need

!pip install -q llama-index llama-index-embeddings-huggingface
!pip install -q transformers accelerate sentence-transformers PyPDF2
!pip install -q langchain langchain-text-splitters
!pip install -q pdf2image pytesseract
!pip install -q gradio
!apt-get -qq install -y poppler-utils tesseract-ocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.6/173.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.7 MB/s eta 0:00:00
Selecting previously unselected package poppler-utils.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.9_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.9) ...
Setting up poppler-utils (24.02.0-1ubuntu9.9) ...
Processing triggers for man-db (2.12.0-4build2) ...


## Core Imports, Embedding Model, and Phi-3-mini-4k-instruct

In [2]:
# Imports and Model Configuration


# Loading an open-source embedding model (for retrieval) and an
# open-source LLM, Phi-3-mini-4k-instruct (for generation)
#
# Phi-3-mini is small enough @ 3.8B params to run on a free Colab
# T4 GPU in fp16 while still following instructions well, balancing
# response speed and quality.
# ============================================

import os
import re
import torch

from PyPDF2 import PdfReader
from pdf2image import convert_from_path
import pytesseract

from langchain_text_splitters import RecursiveCharacterTextSplitter

from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

import gradio as gr

# ---- Device ----
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ---- Open-source embedding model (sentence-transformers based) ----
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)

# ---- Open-source LLM: Phi-3-mini-4k-instruct ----
LLM_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

phi3_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
phi3_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
)

phi3_pipe = pipeline(
    "text-generation",
    model=phi3_model,
    tokenizer=phi3_tokenizer,
    return_full_text=False,
)


def phi3_generate(prompt, max_new_tokens=400):
    """Send a prompt to Phi-3-mini using its chat template and return plain text."""
    messages = [{"role": "user", "content": prompt}]
    formatted = phi3_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    output = phi3_pipe(formatted, max_new_tokens=max_new_tokens, do_sample=False)
    return output[0]["generated_text"].strip()


print("Phi-3-mini-4k-instruct loaded and ready.")


Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Phi-3-mini-4k-instruct loaded and ready.


## UI Theme and Styling

In [3]:
# UI Theme

BRAND_BLUE = "#0070BF"      # primary blue
BRAND_CERULEAN = "#00AFF0"  # accent blue
BRAND_LIGHT_BG = "#F2F9FF"  # light tint for backgrounds

app_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="sky",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
).set(
    button_primary_background_fill=BRAND_BLUE,
    button_primary_background_fill_hover=BRAND_CERULEAN,
    button_primary_text_color="#FFFFFF",
    button_secondary_background_fill=BRAND_CERULEAN,
    button_secondary_background_fill_hover=BRAND_BLUE,
    button_secondary_text_color="#FFFFFF",
    block_title_text_color=BRAND_BLUE,
    block_label_text_color=BRAND_BLUE,
    body_background_fill=BRAND_LIGHT_BG,
    body_background_fill_dark="#0A2A3D",
    border_color_primary=BRAND_CERULEAN,
)

custom_css = f"""
#app_banner {{
    background: linear-gradient(90deg, {BRAND_BLUE} 0%, {BRAND_CERULEAN} 100%);
    padding: 18px 24px;
    border-radius: 10px;
    color: white;
    margin-bottom: 12px;
}}
#app_banner h1 {{
    margin: 0;
    font-size: 1.4em;
}}
#app_banner p {{
    margin: 4px 0 0 0;
    opacity: 0.9;
    font-size: 0.9em;
}}
"""


## Document Processing: OCR-Aware PDF Extraction

Digital PDFs get their text pulled directly. Pages with little or no extractable text (a strong signal they're scanned images) are rendered to an image and run through Tesseract OCR instead.

In [4]:
# OCR-Aware PDF Text Extraction

# Extracting text page-by-page with PyPDF2. If a page comes back
# with almost no text, we treat it as a scanned page, render it to
# an image with pdf2image, and OCR it with pytesseract instead.
#
# A pipeline that only handles digital PDFs breaks the moment
# someone uploads a scanned certificate or a faxed document.


def extract_pdf_pages(file_path, ocr_threshold=30, dpi=200):
    """Extract text from every page, falling back to OCR for scanned pages."""
    reader = PdfReader(file_path)
    pages = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        ocr_used = False

        if len(text.strip()) < ocr_threshold:
            images = convert_from_path(file_path, dpi=dpi, first_page=i + 1, last_page=i + 1)
            if images:
                ocr_text = pytesseract.image_to_string(images[0])
                if len(ocr_text.strip()) > len(text.strip()):
                    text = ocr_text
                    ocr_used = True

        pages.append({"page": i, "text": text, "ocr_used": ocr_used})

    return pages


## Document Type Classification and Boundary Detection

Uses Phi-3-mini to classify each page and detect where one logical document ends and the next begins (needed since some documents span multiple pages).

In [5]:
# Document Type Classification & Boundary Detection


VALID_DOC_TYPES = [
    "Cover Letter", "Certificate Of Quality", "Packaging Component Specification",
    "Animal Origin Statement", "Compliance Bulletin", "Vendor Qualification Record",
    "Certificate of Processing", "BSE/TSE Declaration", "Material Description Sheet",
    "Chain of Custody", "Other"
]

DOC_TYPE_DESCRIPTIONS = {
    "Cover Letter": "an introductory letter or email with 'to:' and 'from:' 'Dear' 'Regards' listing what's included in a document package, not itself containing technical data",
    "Certificate Of Quality": "a quality release document with batch/lot number tables, certificate date, certificate numbers, batch no., manufacture/expiry dates, and test results (dimensions, hardness, sterility) — NOT a sterilization processing certificate or dose audit",
    "Packaging Component Specification": "scope, primary packaging, secondary packaging, revision history, component material / grade, specification / standard",
    "Animal Origin Statement": "a statement confirming a product contains no animal-derived or human-derived materials statement, regulatory basis",
    "Compliance Bulletin": "a document with a table of contents listing the chemical composition, physical properties (hardness, density), and technical bulletins describing what a component is made of and its regulatory compliance",
    "Vendor Qualification Record": "vendor qualification history, vendor name, sub-supplier company names, ISO certification numbers ,qualification status, approved for, regulatory information expiry dates, (ISO numbers), expiry lists, approved product list, and performance metrics",
    "Certificate of Processing": "gamma radiation sterilization / dose audit, process run id, sterilization mode, validated dose range, dosimetry summary",
    "BSE/TSE Declaration": "a declaration confirming products contain no materials of animal origin (bovine, ovine, caprine, porcine), citing BSE/TSE risk assessment regulations and an assessment reference number",
    "Material Description Sheet": "a technical sheet listing materials of construction for a physical component, sterilization compatibility, and physical properties like dimensions, weight, and operating pressure",
    "Chain of Custody": "a document listing manufactured assemblies and the numbered steps of their supply chain from raw material receipt through distribution to the end customer, with a traceability statement",
    "Other": "use only if the page clearly fits none of the types above",
}

def clean_doc_type(response):
    """Clean up the LLM's response into one of the valid labels."""
    cleaned = response.strip().replace('"', '').replace('`', '').replace('*', '').lower().replace(".", "").strip()
    cleaned_title = cleaned.title()
    for label in VALID_DOC_TYPES:
        if label.lower() in cleaned.lower():
            return label
    return cleaned_title


def strip_letterhead(text, max_lines=3):
    """Drop leading boilerplate lines (company name, address, phone, website)
    so document comparisons focus on substantive content, not shared letterhead."""
    lines = text.strip().split("\n")
    kept = []
    boilerplate_patterns = [
        r"^\s*tel:?\s*[\+\(]?\d",           # phone numbers
        r"^\s*www\.",                        # websites
        r"^\s*[\w\s]+,\s*(inc|llc|corp)\.?\s*$",  # "West Pharmaceutical Services, Inc."
        r"\b(street|st\.|road|rd\.|avenue|ave\.)\b.*\d{5}",  # street address w/ zip
        r"^\s*\d+\s+[\w\s]+,\s*\w+\s*,?\s*[A-Z]{2}\s*\d{5}",  # "347 Oliver Street, Jersey Shore, PA 17740"
    ]

    for i, line in enumerate(lines):
        if i < max_lines and any(re.search(p, line, re.IGNORECASE) for p in boilerplate_patterns):
            continue
        kept.append(line)

    return "\n".join(kept).strip()

def classify_document_type(text):
    """Ask Phi-3-mini to label a page with one of the known document types."""
    type_list = "\n".join(f"- {k}: {v}" for k, v in DOC_TYPE_DESCRIPTIONS.items())
    prompt = f"""You are a pharmaceutical document classifier. Based on the page content below, classify it into ONE of these document types:

{type_list}

Page Content:
{text[:1500]}

Respond with ONLY the document type name. No explanation."""
    response = phi3_generate(prompt, max_new_tokens=20)
    return clean_doc_type(response)



# During testing, the model was not stable in how it classified documents:
# processing the same PDF could return different classifications, likely a
# limitation of using a smaller model. This stable version makes several
# classification calls and takes the majority vote, trading extra processing
# time for more consistent labels.

def classify_document_type_stable(text, votes=3):
    results = [classify_document_type(text) for _ in range(votes)]
    return max(set(results), key=results.count)


def get_yes_no_confidence(prompt):
    """
    Run one forward pass and return the model's confidence that the answer is 'Yes',
    as a value between 0 and 1 — read directly from the token probabilities instead
    of sampling multiple full generations.
    """
    messages = [{"role": "user", "content": prompt}]
    formatted = phi3_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = phi3_tokenizer(formatted, return_tensors="pt").to(phi3_model.device)

    with torch.no_grad():
        outputs = phi3_model.generate(
            **inputs,
            max_new_tokens=1,          # we only need the model's very first choice
            do_sample=False,
            output_scores=True,        # gives us the raw probability distribution
            return_dict_in_generate=True,
        )

    logits = outputs.scores[0][0]                  # scores over the vocabulary for that first token
    probs = torch.softmax(logits, dim=-1)

    # Token ids verified for the Phi-3 tokenizer: "Yes" = 3869, "No" = 1939.
    # (Some tokenizers encode "Yes" differently depending on a leading space.)
    yes_id = phi3_tokenizer.encode(" Yes", add_special_tokens=False)[-1]
    no_id = phi3_tokenizer.encode(" No", add_special_tokens=False)[-1]

    yes_prob = probs[yes_id].item()
    no_prob = probs[no_id].item()

    total = yes_prob + no_prob
    return yes_prob / total if total > 0 else 0.5   # normalized so Yes/No sum to 1


def is_same_document_confident(prev_text, curr_text, doc_type=None, threshold=0.65):
    """
    Decide whether two pages belong to the same document using the model's actual
    confidence, not repeated sampling. Requires >= threshold confidence in 'Yes'
    to merge — this replaces is_same_document_stable's majority-vote approach.
    """
    prev_clean = strip_letterhead(prev_text)
    curr_clean = strip_letterhead(curr_text)

    prompt = f"""You are checking whether two consecutive pages from a document package belong to the SAME individual document.
Recognize shared letterhead and / or company info at the beginning of a page. A shared letterhead is NOT evidence of document sameness.
A NEW document starts when the page has a different title/heading, a different document number, or covers a different topic — even if
both pages relate to the same product or share similar section formatting.
Do not treat shared subject matter, shared product references, or similar regulatory language as evidence of sameness. Only an explicit
 continuation ("continued", "page 2 of 2") or a matching document title/number counts as evidence of sameness.

Previous page type: {doc_type or 'unknown'}

Previous Page:
{prev_clean[:800]}

Current Page:
{curr_clean[:800]}

Do these two pages belong to the SAME document? Answer ONLY 'Yes' or 'No'."""

    confidence = get_yes_no_confidence(prompt)
    return confidence >= threshold

## Grouping Pages, Chunking, and Metadata Tagging

Consecutive pages of the same document get merged into one logical document, then split into 512-character chunks (100-character overlap) via `RecursiveCharacterTextSplitter`. Every chunk carries `doc_type`, `chunk_index`, `doc_id`, `source_file`, and page range metadata.

In [6]:
# Group Pages -> Logical Documents -> Chunks with Metadata
#
# This cell turns a list of raw PDF pages into indexed and searchable chunks in three steps:
#
# 1. build_metadata_store(): walks through the pages in order, using classify_document_type_stable()
#    on the first page of each new document and is_same_document_confident() to decide whether each
#    following page continues that same doc type or starts a new one. Every page gets tagged with
#    a doc_id, doc_type, and its position within its document and returns the next unused doc_id (doc_counter + 1)
#    so that a second file processed later in the same session always gets fresh doc_ids
#    instead of colliding with the previous file's last one.
#
# 2. group_into_logical_docs(): merges consecutive pages that share a doc_id into a single logical
#    document record, combining their text and tracking the page range each document spans.
#
# 3. chunk_logical_docs(): splits each logical document's combined text into overlapping ~512-character
#    chunks (RecursiveCharacterTextSplitter), attaching doc_type/doc_id/source_file/page range metadata
#    to every chunk so it can later be filtered and cited correctly at retrieval time.


# Classify each page and assign it a doc_id, grouping consecutive pages
# using confidence-based boundary decisions (instead of a majority vote)
def build_metadata_store(pages, source_file, next_doc_id=0):
    metadata_store = []
    current_doc_type = None
    doc_counter = next_doc_id
    page_counter = 0

    for i, page in enumerate(pages):
        if i == 0:
            current_doc_type = classify_document_type_stable(page["text"])
        else:
            same = is_same_document_confident(pages[i - 1]["text"], page["text"], current_doc_type)
            if not same:
                doc_counter += 1
                current_doc_type = classify_document_type_stable(page["text"])
                page_counter = 0
            else:
                page_counter += 1

        metadata_store.append({
            "page": page["page"],
            "text": page["text"],
            "doc_type": current_doc_type,
            "doc_id": f"doc_{doc_counter:03d}",
            "source_file": source_file,
            "page_in_doc": page_counter,
            "ocr_used": page.get("ocr_used", False),
        })

    return metadata_store, doc_counter + 1  # next unused doc_id for the next uploaded file


def group_into_logical_docs(metadata_store):
    """Merge consecutive pages belonging to the same document into one record."""
    logical_docs = []
    current_doc = None

    for page in metadata_store:
        if page["page_in_doc"] == 0:
            if current_doc is not None:
                logical_docs.append(current_doc)
            current_doc = {
                "text": "",
                "doc_type": page["doc_type"],
                "doc_id": page["doc_id"],
                "source_file": page["source_file"],
                "page_start": page["page"],
                "page_end": page["page"],
                "ocr_used": page["ocr_used"],
            }

        current_doc["text"] += "\n\n" + page["text"]
        current_doc["page_end"] = page["page"]
        current_doc["ocr_used"] = current_doc["ocr_used"] or page["ocr_used"]

    if current_doc is not None:
        logical_docs.append(current_doc)

    return logical_docs


splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)

def chunk_logical_docs(logical_docs):
    """Split each logical document into overlapping chunks and attach metadata to each."""
    chunked_documents = []
    for doc in logical_docs:
        chunks = splitter.split_text(doc["text"])
        for chunk_idx, chunk in enumerate(chunks):
            chunked_documents.append(
                Document(
                    text=chunk,
                    metadata={
                        "doc_type": doc["doc_type"],
                        "chunk_index": chunk_idx,
                        "doc_id": doc["doc_id"],
                        "source_file": doc["source_file"],
                        "page_start": doc["page_start"],
                        "page_end": doc["page_end"],
                        "ocr_used": doc["ocr_used"],
                    }
                )
            )
    return chunked_documents


## Embeddings and Vector Index

Each new document's chunks get embedded and added to a shared LlamaIndex `VectorStoreIndex`, so multiple uploaded files are searchable together.

In [7]:
# Build / Update the Vector Index


rag_index = None
known_doc_types = set()
processed_files = []
_global_doc_counter = 0


def add_document_to_index(file_path, source_file):
    """Run the full extraction -> classification -> chunking pipeline and add to the index."""
    global rag_index, _global_doc_counter

    pages = extract_pdf_pages(file_path)
    metadata_store, _global_doc_counter = build_metadata_store(pages, source_file, _global_doc_counter)
    logical_docs = group_into_logical_docs(metadata_store)
    chunked_docs = chunk_logical_docs(logical_docs)

    for doc in logical_docs:
        known_doc_types.add(doc["doc_type"])
    processed_files.append(source_file)

    if rag_index is None:
        rag_index = VectorStoreIndex.from_documents(chunked_docs)
    else:
        for chunk_doc in chunked_docs:
            rag_index.insert(chunk_doc)

    return {
        "num_pages": len(pages),
        "num_logical_docs": len(logical_docs),
        "num_chunks": len(chunked_docs),
        "doc_types_found": sorted(set(d["doc_type"] for d in logical_docs)),
        "ocr_pages_used": sum(1 for p in pages if p["ocr_used"]),
    }


## Query Routing

Phi-3-mini predicts which document type is most likely to answer the query, restricted to the document types actually present in the index.

In [8]:
# Query Routing


def predict_doc_type_for_query(query):
    """Ask Phi-3-mini which indexed document type is most likely to answer this query."""
    if not known_doc_types:
        return None

    type_list = "\n".join(f"- {t}: {DOC_TYPE_DESCRIPTIONS.get(t, '')}" for t in sorted(known_doc_types))
    prompt = f"""You are an intelligent assistant that routes user queries to the most relevant document type.

User query: "{query}"

Which document type is most likely to contain the answer?
Choose ONLY ONE from:

{type_list}

Respond with ONLY the document type name. No explanation."""
    response = phi3_generate(prompt, max_new_tokens=20)
    predicted = clean_doc_type(response)
    return predicted if predicted in known_doc_types else None

def predict_doc_type_for_query_stable(query, votes=3):
    results = [predict_doc_type_for_query(query) for _ in range(votes)]
    results = [r for r in results if r]
    if not results:
        return None
    return max(set(results), key=results.count)


## Full RAG Pipeline: Retrieval -> Context -> Prompt -> Model -> Answer + Sources

Retrieves chunks filtered by predicted document type (falling back to an unfiltered search if that comes back empty), builds a labeled context block, and prompts Phi-3-mini with explicit grounding and citation instructions. Confidence is derived from retrieval similarity scores.

In [9]:
# Full RAG Answer Pipeline


MODE_INSTRUCTIONS = {
    "Concise": "Answer in 1-3 sentences, as directly as possible.",
    "Detailed": "Provide a thorough, well-structured answer covering all relevant details from the context.",
    "Quote-based": "Answer primarily by quoting short phrases (under 15 words each) directly from the context, with a citation after each quote.",
}


def retrieve_chunks(query, predicted_doc_type_stable, top_k=4):
    """Retrieve chunks filtered by doc_type; fall back to an unfiltered search if empty."""
    if rag_index is None:
        return []

    if predicted_doc_type_stable:
        retriever = rag_index.as_retriever(
            similarity_top_k=top_k,
            filters=MetadataFilters(filters=[
                MetadataFilter(key="doc_type", value=predicted_doc_type_stable, operator=FilterOperator.EQ)
            ])
        )
        nodes = retriever.retrieve(query)
        if nodes:
            return nodes

    retriever = rag_index.as_retriever(similarity_top_k=top_k)
    return retriever.retrieve(query)


def build_context_and_sources(nodes):
    """Build a labeled context block for the prompt, and a parallel list of source metadata."""
    context_parts = []
    sources = []
    for i, node in enumerate(nodes, start=1):
        meta = node.metadata
        label = f"Source {i}"
        context_parts.append(
            f"[{label}] ({meta.get('doc_type')}, pages {meta.get('page_start')}-{meta.get('page_end')})\n{node.text}"
        )
        sources.append({
            "label": label,
            "doc_type": meta.get("doc_type"),
            "source_file": meta.get("source_file"),
            "page_start": meta.get("page_start"),
            "page_end": meta.get("page_end"),
            "score": round(float(node.score), 3) if node.score is not None else None,
        })
    return "\n\n".join(context_parts), sources


def answer_query(query, mode="Concise", top_k=4):
    """Retrieval -> Context Building -> Prompt -> Model -> Answer + Sources."""
    if rag_index is None:
        return {
            "answer": "Please upload and process a document first.",
            "sources": [], "confidence": None, "chunk_count": 0, "predicted_doc_type": None,
        }

    predicted_doc_type = predict_doc_type_for_query_stable(query)
    nodes = retrieve_chunks(query, predicted_doc_type, top_k=top_k)

    if not nodes:
        return {
            "answer": "I cannot find this information in the provided documents.",
            "sources": [], "confidence": 0.0, "chunk_count": 0, "predicted_doc_type": predicted_doc_type,
        }

    context, sources = build_context_and_sources(nodes)
    mode_instruction = MODE_INSTRUCTIONS.get(mode, MODE_INSTRUCTIONS["Concise"])

    prompt = f"""You are a helpful assistant answering questions using ONLY the context provided below.
If the answer is not contained in the context, say "I cannot find this information in the provided documents."
Cite the relevant source label (e.g. [Source 1]) after each claim you make. {mode_instruction}

Context:
{context}

Question: {query}

Answer:"""

    answer = phi3_generate(prompt, max_new_tokens=400)

    scores = [s["score"] for s in sources if s["score"] is not None]
    confidence = round(sum(scores) / len(scores) * 100, 1) if scores else None

    return {
        "answer": answer,
        "sources": sources,
        "confidence": confidence,
        "chunk_count": len(nodes),
        "predicted_doc_type": predicted_doc_type,
    }


## Gradio UI

Themed interface: upload one or more PDFs (digital or scanned), watch live status while they're processed, then chat and see each answer's sources, confidence, and chunk count alongside it.

In [10]:
# Build the Chatbot UI


def format_bot_reply(result):
    """Turn a result dict into a chat-friendly markdown reply with sources and confidence."""
    lines = [result["answer"], ""]
    if result["sources"]:
        lines.append(
            f"**Confidence:** {result['confidence']}%  |  **Chunks used:** {result['chunk_count']}  |  **Routed to:** {result['predicted_doc_type']}"
        )
        lines.append("")
        lines.append("**Sources:**")
        for s in result["sources"]:
            lines.append(
                f"- [{s['label']}] {s['doc_type']} — {s['source_file']}, pages {s['page_start']}-{s['page_end']} (score: {s['score']})"
            )
    return "\n".join(lines)


def process_documents(files):
    """Index each uploaded file and return a status line plus per-file details."""
    if not files:
        return "No files uploaded.", ""

    summary_lines = []
    for f in files:
        source_name = os.path.basename(f.name)
        stats = add_document_to_index(f.name, source_name)
        summary_lines.append(
            f"**{source_name}**: {stats['num_pages']} page(s), {stats['num_logical_docs']} document(s), "
            f"{stats['num_chunks']} chunk(s), {stats['ocr_pages_used']} OCR page(s) — types: {', '.join(stats['doc_types_found'])}"
        )

    status = f"Processed {len(files)} file(s)."
    details = "\n\n".join(summary_lines)
    return status, details


def handle_chat(message, history, mode):
    if not message.strip():
        return history

    result = answer_query(message, mode=mode)
    reply = format_bot_reply(result)

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": reply}
    ]
    return history


def save_chat_history(history):
    """Extra feature: export the conversation to a downloadable .txt file."""
    if not history:
        return None
    lines = []
    for msg in history:
        speaker = "You" if msg["role"] == "user" else "Bot"
        lines.append(f"{speaker}: {msg['content']}")
        lines.append("")
    path = "/tmp/chat_history.txt"
    with open(path, "w") as f:
        f.write("\n".join(lines))
    return path


with gr.Blocks(title="Pharma Document Q&A") as demo:

    gr.HTML(
        """
        <div id="app_banner">
            <h1>Pharma Document Q&amp;A</h1>
            <p>Upload documents, ask questions, and get grounded answers with sources and confidence
            all powered by an open-source RAG pipeline (Phi-3-mini-4k-instruct).</p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("#### Documents")
            pdf_input = gr.File(label="Upload PDF(s)", file_types=[".pdf"], file_count="multiple")
            process_btn = gr.Button("Process Documents", variant="primary")
            doc_status = gr.Markdown("_No documents processed yet._")
            doc_details = gr.Markdown("")

        with gr.Column(scale=2):
            gr.Markdown("#### Chat")
            mode_dropdown = gr.Dropdown(
                choices=["Concise", "Detailed", "Quote-based"],
                value="Concise",
                label="Answer mode"
            )
            chatbot = gr.Chatbot(label="Chat History", height=380)
            bot_status = gr.Markdown("")
            user_input = gr.Textbox(
                placeholder="Ask a question about your uploaded documents...",
                label="Your Question"
            )
            with gr.Row():
                send_btn = gr.Button("Send", variant="primary")
                clear_btn = gr.Button("Clear Chat", variant="secondary")
            with gr.Row():
                save_btn = gr.Button("Save Chat History")
                download_file = gr.File(label="Download", visible=True)

    process_btn.click(
        lambda: "Processing document(s)... (OCR runs automatically on scanned pages)", None, doc_status
    ).then(
        process_documents, inputs=pdf_input, outputs=[doc_status, doc_details]
    )

    send_btn.click(
        lambda: "Retrieving and generating answer...", None, bot_status
    ).then(
        handle_chat, inputs=[user_input, chatbot, mode_dropdown], outputs=chatbot
    ).then(
        lambda: "Ready", None, bot_status
    ).then(
        lambda: "", None, user_input
    )

    user_input.submit(
        lambda: "Retrieving and generating answer...", None, bot_status
    ).then(
        handle_chat, inputs=[user_input, chatbot, mode_dropdown], outputs=chatbot
    ).then(
        lambda: "Ready", None, bot_status
    ).then(
        lambda: "", None, user_input
    )

    clear_btn.click(lambda: ([], ""), None, [chatbot, bot_status])
    save_btn.click(save_chat_history, inputs=chatbot, outputs=download_file)


In [11]:
# Launch the app. Gradio's share=True creates a temporary public link (needed on Colab).
gr.close_all()
demo.launch(share=True, theme=app_theme, css=custom_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b04a490789f2b20dfc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [25]:
# Reprocess a sample PDF from scratch, using the current classification rules
# (VALID_DOC_TYPES / DOC_TYPE_DESCRIPTIONS) and boundary-detection logic.
# Resets the index and counters first so no stale classifications carry over
# from a previous run. This is the GPU-heavy step, since every page runs
# through document classification and boundary detection.
#
# The sample PDFs live in sample_docs/ in the repo. Upload them to /content/ first.
#   "A" -> supplier-package-a.pdf  (digital PDF, text layer)
#   "B" -> supplier-package-b.pdf  (scanned PDF, exercises the OCR path)

SAMPLE_DIR = "/content"
SAMPLES = {"A": "supplier-package-a.pdf", "B": "supplier-package-b.pdf"}
ACTIVE_SAMPLE = "B"

rag_index = None
known_doc_types = set()
processed_files = []
_global_doc_counter = 0

sample_file = SAMPLES[ACTIVE_SAMPLE]
stats = add_document_to_index(f"{SAMPLE_DIR}/{sample_file}", sample_file)
print(stats)

[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

{'num_pages': 7, 'num_logical_docs': 6, 'num_chunks': 9, 'doc_types_found': ['BSE/TSE Declaration', 'Certificate Of Quality', 'Chain of Custody', 'Material Description Sheet', 'Packaging Component Specification'], 'ocr_pages_used': 7}


In [26]:
# Sanity check: show the metadata of the top chunk for one query
import json

for node in rag_index.as_retriever(similarity_top_k=1).retrieve("What is the validated dose range for gamma radiation sterilization?"):
    print(json.dumps(node.metadata, indent=2))

{
  "doc_type": "Material Description Sheet",
  "chunk_index": 1,
  "doc_id": "doc_000",
  "source_file": "supplier-package-b.pdf",
  "page_start": 0,
  "page_end": 0,
  "ocr_used": true
}


In [27]:
# Test set for supplier-package-a.pdf (digital, 9 pages, 7 documents)
#   doc_000 p0    Cover Letter
#   doc_001 p1-2  Certificate of Quality
#   doc_002 p3    Packaging Component Specification
#   doc_003 p4    Animal Origin Statement
#   doc_004 p5    Compliance Bulletin
#   doc_005 p6-7  Vendor Qualification Record
#   doc_006 p8    Certificate of Processing

test_set_a = [
    {
        "query": "What documents are included in this supplier documentation package?",
        "expected_answer": """1. Certificate of Quality, 2. Packaging Component Specification,
        3. Animal Origin Statement, 4. Compliance Bulletin, 5. Vendor Qualification Record,
        6. Certificate of Processing""",
        "correct_doc_id": "doc_000",
        "correct_page_range": (0, 0),
    },
    {
        "query": "What is the certificate date for certificate of quality number CQ-7714032-0005?",
        "expected_answer": "4 March 2025",
        "correct_doc_id": "doc_001",
        "correct_page_range": (1, 2),
    },
    {
        "query": "What is the bioburden result in the Certificate of Quality?",
        "expected_answer": "2 CFU/stopper",
        "correct_doc_id": "doc_001",
        "correct_page_range": (1, 2),
    },
    {
        "query": "What are the primary packaging components?",
        "expected_answer": "Inner bag, bag closure, desiccant sachet",
        "correct_doc_id": "doc_002",
        "correct_page_range": (3, 3),
    },
    {
        "query": "What is the document number of the Animal Origin Statement?",
        "expected_answer": "NCS-AOS-2025-004",
        "correct_doc_id": "doc_003",
        "correct_page_range": (4, 4),
    },
    {
        "query": "What standards does the NCS-2210 Grey Stopper comply with?",
        "expected_answer": "USP <381>, USP <87>, EP 3.2.9, and ISO 8362-2",
        "correct_doc_id": "doc_004",
        "correct_page_range": (5, 5),
    },
    {
        "query": "What is the certificate number for ISO 9001:2015?",
        "expected_answer": "QMS-NB-2023-0412",
        "correct_doc_id": "doc_005",
        "correct_page_range": (6, 7),
    },
    {
        "query": "What material does the sub-supplier Calloway Polymers Ltd. provide?",
        "expected_answer": "Bromobutyl rubber (BIIR)",
        "correct_doc_id": "doc_005",
        "correct_page_range": (6, 7),
    },
    {
        "query": "What is the process run ID for gamma radiation sterilization?",
        "expected_answer": "GRX-2025-0447B",
        "correct_doc_id": "doc_006",
        "correct_page_range": (8, 8),
    },
    {
        "query": "What is the validated dose range for gamma radiation sterilization?",
        "expected_answer": "25.0 kGy (min) - 45.0 kGy (max)",
        "correct_doc_id": "doc_006",
        "correct_page_range": (8, 8),
    },
]

In [28]:
# Test set for supplier-package-b.pdf (scanned, 7 pages, 6 documents)
#   doc_000 p0    Material Description Sheet
#   doc_001 p1    Certificate of Quality (Low Flow Kit)
#   doc_002 p2    Certificate of Quality (High Flow Kit)  <- same type, new document
#   doc_003 p3-4  Packaging Component Specification
#   doc_004 p5    BSE/TSE Declaration
#   doc_005 p6    Chain of Custody

test_set_b = [
    {
        "query": "What is the operating temperature range for FlowLine single-use kits?",
        "expected_answer": "+4 C to +37 C",
        "correct_doc_id": "doc_000",
        "correct_page_range": (0, 0),
    },
    {
        "query": "What is the maximum operating pressure for FlowLine kits?",
        "expected_answer": "3.0 bar",
        "correct_doc_id": "doc_000",
        "correct_page_range": (0, 0),
    },
    {
        "query": "What is the lot number for the FlowLine Low Flow Kit?",
        "expected_answer": "HB-77310",
        "correct_doc_id": "doc_001",
        "correct_page_range": (1, 1),
    },
    {
        "query": "What is the lot number for the FlowLine High Flow Kit?",
        "expected_answer": "HB-77452",
        "correct_doc_id": "doc_002",
        "correct_page_range": (2, 2),
    },
    {
        "query": "What material is used for the blister tray?",
        "expected_answer": "PETG, 0.8 mm",
        "correct_doc_id": "doc_003",
        "correct_page_range": (3, 4),
    },
    {
        "query": "What material is the outer pouch made of?",
        "expected_answer": "Polyethylene / nylon laminate",
        "correct_doc_id": "doc_003",
        "correct_page_range": (3, 4),
    },
    {
        "query": "What is the assessment reference for the BSE/TSE risk assessment?",
        "expected_answer": "TSE-RA-2024-FL-003",
        "correct_doc_id": "doc_004",
        "correct_page_range": (5, 5),
    },
    {
        "query": "Who prepared the Global Chain of Custody document?",
        "expected_answer": "Supply Chain Quality, Halvorsen Bioprocess AB",
        "correct_doc_id": "doc_005",
        "correct_page_range": (6, 6),
    },
    {
        "query": "Where does cleanroom assembly and packaging take place?",
        "expected_answer": "Tallinn, Estonia",
        "correct_doc_id": "doc_005",
        "correct_page_range": (6, 6),
    },
    {
        "query": "How is each kit made traceable?",
        "expected_answer": "Each kit carries a unique lot number traceable to raw material lots, assembly date, and cleanroom operator records.",
        "correct_doc_id": "doc_005",
        "correct_page_range": (6, 6),
    },
]

In [29]:
# Runs a query while timing routing, retrieval, and generation separately

import time

def run_instrumented_query(query, mode="Concise", top_k=4):
    timings = {}

    t0 = time.time()
    predicted_doc_type = predict_doc_type_for_query_stable(query)
    timings["routing_sec"] = time.time() - t0

    t0 = time.time()
    nodes = retrieve_chunks(query, predicted_doc_type, top_k=top_k)
    timings["retrieval_sec"] = time.time() - t0

    if not nodes:
        return {
            "query": query, "predicted_doc_type": predicted_doc_type,
            "retrieved": [], "answer": "I cannot find this information in the provided documents.",
            "sources": [], "timings": timings,
        }

    context, sources = build_context_and_sources(nodes)
    mode_instruction = MODE_INSTRUCTIONS.get(mode, MODE_INSTRUCTIONS["Concise"])
    prompt = f"""You are a helpful assistant answering questions using ONLY the context provided below.
If the answer is not contained in the context, say "I cannot find this information in the provided documents."
Cite the relevant source label (e.g. [Source 1]) after each claim you make. {mode_instruction}

Context:
{context}

Question: {query}

Answer:"""

    t0 = time.time()
    answer = phi3_generate(prompt, max_new_tokens=400)
    timings["generation_sec"] = time.time() - t0

    retrieved_info = [
        {"doc_id": n.metadata.get("doc_id"), "page_start": n.metadata.get("page_start"), "page_end": n.metadata.get("page_end")}
        for n in nodes
    ]

    return {
        "query": query, "predicted_doc_type": predicted_doc_type,
        "retrieved": retrieved_info, "answer": answer, "sources": sources, "timings": timings,
    }

In [30]:
# Use the test set that matches the PDF processed above (ACTIVE_SAMPLE)
test_set = test_set_a if ACTIVE_SAMPLE == "A" else test_set_b

In [31]:
# Run every test query

results = []
for item in test_set:
    r = run_instrumented_query(item["query"])
    r["expected_answer"] = item["expected_answer"]
    r["correct_doc_id"] = item["correct_doc_id"]
    r["correct_page_range"] = item["correct_page_range"]
    results.append(r)
    print(f"Ran: {item['query'][:60]}...")

[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/doc

Ran: What is the operating temperature range for FlowLine single-...


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/doc

Ran: What is the maximum operating pressure for FlowLine kits?...


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/doc

...

Ran: Who prepared the Global Chain of Custody document?...


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/doc

Ran: Where does cleanroom assembly and packaging take place?...


[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ran: How is each kit made traceable?...


In [32]:
# Retrieval metrics: Recall@K, Precision@K, MRR

def is_relevant(retrieved_chunk, correct_doc_id, correct_page_range):
    if retrieved_chunk["doc_id"] != correct_doc_id:
        return False
    lo, hi = correct_page_range
    return not (retrieved_chunk["page_end"] < lo or retrieved_chunk["page_start"] > hi)

recall_hits = 0
precision_scores = []
reciprocal_ranks = []

for r in results:
    flags = [is_relevant(c, r["correct_doc_id"], r["correct_page_range"]) for c in r["retrieved"]]
    recall_hits += any(flags)
    precision_scores.append(sum(flags) / len(flags) if flags else 0)
    rank = next((i + 1 for i, f in enumerate(flags) if f), None)
    reciprocal_ranks.append(1 / rank if rank else 0)

print(f"Recall@K:    {recall_hits / len(results):.2%}")
print(f"Precision@K: {sum(precision_scores) / len(precision_scores):.2%}")
print(f"MRR:         {sum(reciprocal_ranks) / len(reciprocal_ranks):.3f}")

Recall@K:    80.00%
Precision@K: 70.00%
MRR:         0.750


In [33]:
# Inspect which document type was assigned to each doc_id
seen = set()
for node_id, node in rag_index.docstore.docs.items():
    doc_id = node.metadata.get("doc_id")
    doc_type = node.metadata.get("doc_type")
    if doc_id not in seen:
        seen.add(doc_id)
        print(f"{doc_id} | {doc_type}")

doc_000 | Material Description Sheet
doc_001 | Certificate Of Quality
doc_002 | Certificate Of Quality
doc_003 | Packaging Component Specification
doc_004 | BSE/TSE Declaration
doc_005 | Chain of Custody


In [34]:
# Manual review: grade each answer and its citation
correct_flags, citation_flags = [], []

for r in results:
    print("=" * 70)
    print("Q:", r["query"])
    print("Expected:", r["expected_answer"])
    print("Got:", r["answer"])
    print("Cited sources:", [s["doc_type"] for s in r["sources"]])
    print("Retrieved:", [(c["doc_id"], c["page_start"], c["page_end"]) for c in r["retrieved"]])
    correct_flags.append(input("Answer correct? (y/n): ").strip().lower() == "y")
    citation_flags.append(input("Citation accurate? (y/n): ").strip().lower() == "y")

print(f"\nEnd-to-End Accuracy: {sum(correct_flags) / len(correct_flags):.1%}")
print(f"Citation Accuracy:   {sum(citation_flags) / len(citation_flags):.1%}")


Q: What is the operating temperature range for FlowLine single-use kits?
Expected: +4 C to +37 C
Got: The operating temperature range for FlowLine single-use kits is +4 C to +37 C. [Source 1]
Cited sources: ['Material Description Sheet', 'Material Description Sheet']
Retrieved: [('doc_000', 0, 0), ('doc_000', 0, 0)]
Answer correct? (y/n): y
Citation accurate? (y/n): y
Q: What is the maximum operating pressure for FlowLine kits?
Expected: 3.0 bar
Got: The maximum operating pressure for FlowLine kits is 3.0 bar [Source 1].
Cited sources: ['Material Description Sheet', 'Material Description Sheet']
Retrieved: [('doc_000', 0, 0), ('doc_000', 0, 0)]
Answer correct? (y/n): y
Citation accurate? (y/n): y
Q: What is the lot number for the FlowLine Low Flow Kit?
Expected: HB-77310
Got: The lot number for the FlowLine Low Flow Kit is HB-77310. [Source 2]
Cited sources: ['Certificate Of Quality', 'Certificate Of Quality']
Retrieved: [('doc_002', 2, 2), ('doc_001', 1, 1)]
Answer correct? (y/n): y
C

In [35]:
# Average latency per pipeline stage
import statistics

routing_times = [r["timings"]["routing_sec"] for r in results]
retrieval_times = [r["timings"]["retrieval_sec"] for r in results]
generation_times = [r["timings"]["generation_sec"] for r in results if "generation_sec" in r["timings"]]

print(f"Avg routing time:    {statistics.mean(routing_times):.2f}s")
print(f"Avg retrieval time:  {statistics.mean(retrieval_times):.2f}s")
print(f"Avg generation time: {statistics.mean(generation_times):.2f}s")
print(f"Max generation time: {max(generation_times):.2f}s")

Avg routing time:    1.16s
Avg retrieval time:  0.02s
Avg generation time: 1.24s
Max generation time: 1.73s


In [36]:
# Debug: preview the chunks stored for one doc_id
for node_id, node in rag_index.docstore.docs.items():
    if node.metadata.get("doc_id") == "doc_001":
        print(f"pages {node.metadata.get('page_start')}-{node.metadata.get('page_end')}")
        print(node.text[:200])
        print("---")

pages 1-1
Halvorsen Bioprocess AB

Kvarnvagen 12, 752 28 Uppsala, Sweden

Certificate of Quality
Document No.: HQ-2024-1183

Field Value
| Product FlowLine Low Flow Kit
Lot Number HB-77310

Certificate Date

9 
---
